# Evaluating an Exam Using Regular Expressions

This notebook shows how we can use the built-in `re` module
to parse and process text. Our goal is to implement a program that can be used to evaluate the results of an exam. Assume the result of an exam is stored in the string `data` that is defined below:

In [ ]:
data = '''Class: Advanced Witchcraft
          Group: TINF22AI1
          MaxPoints = 60
   
          Exercise:      1. 2. 3. 4. 5. 6.
          Jim Smith:     9 12 10  6  6  0
          John Slow:     4  4  2  0  -  -
          Susi Sorglos:  9 12 12  9  9  6
          1609922:       7  4 12  5  5  3
       '''

This data shows that there has been an exam with the subject <em style="color:blue">Algorithms and Complexity</em>
in the group <em style="color:blue">TINF22AI1</em>. Furthermore, the equation
```
   MaxPoints = 60
```
shows that in order to achieve the best mark, <em style="color:blue">60</em> points would have been necessary.
    
There have been 6 different exercises in this exam and, in this small example, only four students took part, namely *Jim Smith*, *John Slow*, *Susi Sorglos*, and some student that is only represented by their matriculation number. Each of the rows describing the results of the students begins with the name (or matriculation number) of the student followed by the number of points that they have achieved in the different exercises. Our goal is to write a program that is able to compute the marks for all students.

## Imports

We will process the text using Python's standard *regular expressions* that are found in the
`re` module.

In [ ]:
import re

## Auxiliary Functions

The function `mark(max_points, points)` takes two arguments:
- `max_points` is the number of points that need to be achieved in order to get the best mark of $1.0$.
- `points` is the number of points achieved by the student whose mark is to be computed.
  
It is assumed that the relation between the mark of an exam and the number of points achieved in this exam is mostly linear and that a student who has achieved $50\%$ of `max_points` points will get the mark $4.0$, while a student who has achieved $100\%$ of `max_points` points will get the mark $1.0$. Therefore, the formula to calculate the grade is as follows:
$$\textrm{grade} = 7 - 6 \cdot \frac{\texttt{points}}{\texttt{max_points}}$$
However, the worst mark is $5.0$. Therefore, if the mark would fall below that line, the `min` function assures that it is less or equal to $5.0$. Furthermore, the resulting number is rounded to one digit.

In [ ]:
def mark(max_points: int, points: int) -> float:
    grade = 7 - 6 * points / max_points
    return round(min(5.0, grade), 1)

Let's test this function by plotting it. To do this we have to install `matplotlib`.  You can
deactivate the following cell if `matplotlib`is already installed.

In [ ]:
!pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
max_points = 60
points = [points for points in range(max_points + 1)]
grades = [mark(max_points, points) for points in range(max_points + 1)]

plt.figure(figsize=(9, 6))
plt.plot(points, grades, marker='o', linestyle='-')
plt.title('Grade as a Function of Points (Max Points = 60)')
plt.xlabel('Points')
plt.ylabel('Grade')
plt.grid(True)
plt.show()

## Parsing the Data with Regular Expressions

Rather than tokenizing the entire string into a stream, we can process the input functionally line by line. We have two main tasks:
1. Extract the maximum number of points available from the document metadata.
2. Identify the student records, extract their scores, and compute their final marks.

### Extracting Maximum Points

To evaluate the exam, we must first isolate the maximum points definition. We can search the entire input string for the pattern `MaxPoints = [number]`. We use `\s*` to allow for arbitrary spacing around the equals sign, and capture the actual numeric value in a group so it can be extracted and cast to an integer.

In [ ]:
def get_max_points(text: str) -> int:
    match = re.search(r'MaxPoints\s*=\s*([1-9][0-9]*)', text)
    if match:
        return int(match.group(1))
    raise ValueError("MaxPoints definition could not be found in the provided data.")

In [ ]:
max_points = get_max_points(data)
max_points

### Processing Student Records

Next, we need a function that evaluates each student.

A student line consists of an identifier followed by a colon. This identifier can either be a name (allowing upper/lower case letters, spaces, and hyphens) or a $7$-digit matriculation number. After the colon, a sequence of points is provided.

We can define two regular expressions for this task:
- `student_pattern` matches the beginning of a relevant line, capturing the student identifier in the first group and the remaining scores string in the second group.
- `points_pattern` is used to find all valid natural numbers (including zero) within the scores string, effectively ignoring hyphens that signify missed exercises.

In [ ]:
def evaluate_students(text: str, max_points: int) -> None:
    # Matches an optional starting space, the identifier (name or 7-digit ID), 
    # the colon, and the rest of the line containing the scores.
    student_pattern = re.compile(r'^\s*([A-Za-z \-]+|[0-9]{7}):\s*(.*)$')
    # Matches a zero, or any natural number not starting with zero.
    points_pattern = re.compile(r'0|[1-9][0-9]*')
    # Process the text functionally line by line
    for line in text.strip().split('\n'):
        line = line.strip()
        # We skip empty lines and any header/metadata rows that do not contain student data
        if not line or line.startswith(('Class:', 'Group:', 'MaxPoints', 'Exercise:')):
            continue
        match = student_pattern.match(line)
        if match:
            name = match.group(1).strip()
            scores_str = match.group(2)
            # Extract all valid numeric points, ignoring hyphens, and sum them
            points = sum(int(p) for p in points_pattern.findall(scores_str))
            grade = mark(max_points, points)
            print(f'{name} has {points:2d} points and achieved the mark {grade}.')
        else:
            print(f"Warning: Could not parse line -> '{line}'")

### Running the Evaluation

Finally, we execute our parsing logic by passing in the original document string and the maximum points we extracted earlier.

In [ ]:
evaluate_students(data, max_points)